# Team Model Job Template (หนึ่งงาน = หนึ่งโมเดล)

ใช้ notebook นี้เป็น template กลางสำหรับทีม 6–7 คน แต่ละครั้งที่รันจะรับ **หนึ่งงานจาก 41 งาน** และเทรนเพียงหนึ่งโมเดล

## วิธีรับงาน

แก้ค่าใน Cell 2 ได้สองแบบ:

1. **รับตามรอบอัตโนมัติ:** ตั้ง `JOB_ID = None` แล้วใส่ `TEAM_SIZE`, `MEMBER_ID`, `ROUND`
2. **หยิบงานว่างเอง:** ดูตารางงานใน Cell 5 แล้วตั้ง `JOB_ID` เป็นเลข 1–41 ที่ยังไม่มีคนรับ

สูตรแจกตามรอบคือ `(ROUND - 1) * TEAM_SIZE + MEMBER_ID` เช่น ทีม 7 คน สมาชิก 3 รอบ 2 จะได้งาน 10 เมื่อทุกคนจบรอบหนึ่งแล้วให้เพิ่ม `ROUND` และ Run All ใหม่เพื่อรับโมเดลถัดไป

งาน 1 คือ Custom CNN baseline และงาน 2–41 คือ 40 model families ที่คัดจาก Notebook 02 รวมครบ 41 โมเดลโดยไม่ซ้ำ ผลแยกตาม `team-N/member-XX/job-YY` อัตโนมัติ

ทุกคนต้องใช้ repository, split กลาง, `MODE` และสูตรฝึกเดียวกัน ก่อนหยิบงานเองให้บันทึกชื่อผู้รับในตารางแชร์ของทีมเพื่อไม่ให้สองคนเลือก `JOB_ID` เดียวกัน

ลำดับใช้งาน:

1. ติดตั้ง `requirements-training.txt` และเลือก kernel ของโปรเจกต์
2. ตั้ง `.env` สำหรับข้อมูลจาก Drive และรับ split กลางไว้ใน `data/splits`
3. กรอกชื่อและเลือกงานใน Cell 2
4. กด **Run All** เพื่อเทรนโมเดลเดียว
5. ส่งโฟลเดอร์ผลที่ Cell สุดท้ายพิมพ์ แล้วจึงรับรอบหรืองานใหม่

In [ ]:
from pathlib import Path
import sys

# ===== แต่ละคนแก้ส่วนนี้ก่อนรับงาน =====
OWNER = "your_name"
TEAM_SIZE = 7  # จำนวนคนที่กำลังช่วยกันรอบนี้: 6 หรือ 7
MEMBER_ID = 1  # หมายเลขของตัวเอง ไม่ซ้ำกัน: 1..TEAM_SIZE
ROUND = 1  # รอบ 1, 2, 3... แต่ละรอบแต่ละคนได้หนึ่งโมเดล
JOB_ID = None  # None = แจกตาม ROUND; หรือใส่เลข 1..41 เพื่อหยิบงานว่างเอง
SHOW_ALL_JOBS = False  # True = แสดงตารางอ้างอิงครบ 41 งานใน Cell 5

# ===== ตั้งค่า path เฉพาะเมื่อเครื่องนั้นจำเป็น =====
MODE = "quick"  # ทุกงานในรอบเปรียบเทียบต้องใช้ mode เดียวกัน
REPO_PATH = ""  # Colab: path ของ repository ที่ clone/upload มาทั้งชุด
DATA_PATH = "data/raw/clean_32x32"  # ชุดข้อมูลกลาง 32x32; ระบบจะแตก ZIP ให้อัตโนมัติ
SPLIT_PATH = ""  # ว่าง = data/splits/clean_32x32; ทุกคนต้องใช้ split กลางชุดเดียวกัน
OUTPUT_ROOT = ""  # Colab: โฟลเดอร์บน Drive สำหรับเก็บผลถาวร
DOWNLOAD_IF_MISSING = True
DEVICE = "auto"

# ===== สูตรกลาง ห้ามเปลี่ยนรายคน =====
# หากต้องเปลี่ยนค่าเหล่านี้ ให้ตกลงทั้งทีมแล้วเริ่มรอบใหม่พร้อมกัน
SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
PAD_VALUE = 255
EXPECTED_CLASSES = 72
LABEL_LEVEL = 1  # clean_32x32/<หมวด>/<คลาส>/<ภาพ>


In [ ]:
import gc
import importlib.metadata
import json
import subprocess
from dataclasses import asdict, replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
from IPython.display import display
from PIL import Image

folders = [Path(REPO_PATH).expanduser()] if REPO_PATH else [Path.cwd(), *Path.cwd().parents]
project_dir = next((p.resolve() for p in folders if (p / "src/train.py").is_file()), None)
if project_dir is None:
    raise FileNotFoundError("ตั้ง REPO_PATH เป็น repo ที่มี src/train.py")
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))
from src.audit import DEFAULT_EXTENSIONS, audit_dataset, image_paths
from src.paths import configured_data_dir, prepare_image_directory
from src.split import fingerprint, prepare_split, read_rows
from src.train import TrainConfig, build_model, choose_device, fit, load_split, save_json, make_transform
from src.search import rank_runs


TOTAL_JOBS = 41
if not OWNER.strip() or OWNER == "your_name":
    raise ValueError("กรอก OWNER เป็นชื่อผู้รันก่อน")
if TEAM_SIZE not in (6, 7) or not 1 <= MEMBER_ID <= TEAM_SIZE:
    raise ValueError("ใช้ TEAM_SIZE 6 หรือ 7 และ MEMBER_ID ระหว่าง 1..TEAM_SIZE")
if not isinstance(ROUND, int) or isinstance(ROUND, bool) or ROUND < 1:
    raise ValueError("ROUND ต้องเป็นจำนวนเต็มตั้งแต่ 1 ขึ้นไป")
if JOB_ID is not None and (not isinstance(JOB_ID, int) or isinstance(JOB_ID, bool) or not 1 <= JOB_ID <= TOTAL_JOBS):
    raise ValueError("JOB_ID ต้องเป็น None หรือจำนวนเต็ม 1..41")
if MODE not in {"quick", "full"}:
    raise ValueError("Template ทีมใช้ข้อมูลจริงเท่านั้น: MODE quick หรือ full")

assigned_job_id = JOB_ID if JOB_ID is not None else (ROUND - 1) * TEAM_SIZE + MEMBER_ID
if assigned_job_id > TOTAL_JOBS:
    raise ValueError(f"รอบ {ROUND} สมาชิก {MEMBER_ID} ไม่มีงานแล้ว; งานทั้งหมดมี {TOTAL_JOBS} งาน")

epochs = {"quick": 6, "full": 12}[MODE]
output_base = Path(OUTPUT_ROOT).expanduser().resolve() if OUTPUT_ROOT else project_dir / "results/team_family_search"
results_dir = output_base / f"team-{TEAM_SIZE}" / f"member-{MEMBER_ID:02d}" / f"job-{assigned_job_id:02d}"
results_dir.mkdir(parents=True, exist_ok=True)
assignment_mode = "manual" if JOB_ID is not None else "round"
print("ผู้รัน:", OWNER, "สมาชิก:", MEMBER_ID, "จาก", TEAM_SIZE)
print("รับงาน:", assigned_job_id, "แบบ", assignment_mode, "รอบ", ROUND)
print("torch", torch.__version__, "timm", timm.__version__, "device", choose_device(DEVICE))

## 1. เลือกงานที่รับผิดชอบ

Cell ถัดไปสร้างตาราง 41 งานจาก Custom CNN และรายชื่อ 40 families ที่คัดใน Notebook 02 แล้วเลือกมาเพียงหนึ่งแถวตาม `JOB_ID`

ถ้าใช้การแจกตามรอบ ไม่ต้องเลือกชื่อโมเดลเอง ถ้าหยิบงานว่างเอง ให้เช็กตารางแชร์ของทีมก่อนและจองหมายเลขงานทันที

ตรวจชื่อ pretrained weights ก่อนฝึก หากโหลดไม่ได้จะบันทึก failure ตามจริงและไม่เปลี่ยนเป็น random weights

In [ ]:
team_plan = json.loads((project_dir / "configs/experiments/team_family_assignments.json").read_text())
custom_job = {
    "family": "CustomCNN",
    "model": "custom_cnn",
    "group": "From scratch",
    "tier": "baseline",
    "cost": "low",
    "hypothesis": "baseline CNN ที่อธิบาย convolution/pooling/loss ได้ทุกชั้น",
}
task_table = pd.DataFrame([custom_job, *team_plan["candidates"]])
task_table.insert(0, "job_id", range(1, len(task_table) + 1))
if len(task_table) != TOTAL_JOBS or task_table["family"].duplicated().any():
    raise ValueError("รายการงานต้องมี 41 โมเดลและ family ต้องไม่ซ้ำ")

selected = task_table[task_table.job_id == assigned_job_id].copy()
selected_job = selected.iloc[0].to_dict()
model_source = "โค้ดใน src/train.py (ฝึกจากศูนย์)" if selected_job["model"] == "custom_cnn" else "โมเดล pretrained จาก timm"
model_detail = (
    "Conv 4 blocks: 32→64→128→256, AdaptiveAvgPool, Dropout, Linear 72 classes"
    if selected_job["model"] == "custom_cnn"
    else f"ชื่อ architecture ที่ส่งให้ timm.create_model: {selected_job['model']}"
 )
selected_summary = pd.Series({
    "หมายเลขงาน": assigned_job_id,
    "ชื่อ family": selected_job["family"],
    "รหัส architecture": selected_job["model"],
    "แหล่งโมเดล": model_source,
    "โครงสร้าง": model_detail,
    "กลุ่ม": selected_job["group"],
    "ระดับต้นทุน": selected_job["cost"],
    "สมมติฐาน": selected_job["hypothesis"],
}, name="โมเดลที่ Run All ครั้งนี้จะเทรน")
display(selected_summary)

if SHOW_ALL_JOBS:
    print("ตารางอ้างอิง 41 งานด้านล่างไม่ใช่รายการที่จะเทรนพร้อมกัน")
    display(task_table[["job_id", "family", "model", "group", "cost"]].rename(columns={
        "job_id": "หมายเลขงาน", "family": "ชื่อ family", "model": "รหัส architecture",
        "group": "กลุ่ม", "cost": "ระดับต้นทุน",
    }))
else:
    print("ตั้ง SHOW_ALL_JOBS=True ใน Cell 2 หากต้องการดูรายชื่อครบ 41 งาน")

## 2. ตรวจข้อมูลจริงและ split กลาง

โหลดข้อมูลจาก Drive เฉพาะเมื่อยังไม่มี cache ใช้ภาพจริงทั้งหมดตาม split กลาง
ตรวจข้อมูลกับ hash เดิมก่อนเทรน และแสดงภาพตัวอย่างด้านล่าง ไม่มีการสุ่มสร้าง dataset
หากแจ้งว่า split หาย ให้รับ `train.csv`, `val.csv`, `label_to_index.json`, `split_meta.json` จากผู้ดูแลข้อมูล

In [ ]:
requested_data = Path(DATA_PATH).expanduser() if DATA_PATH else configured_data_dir()
data_dir = (requested_data if requested_data.is_absolute() else project_dir / requested_data).resolve()
data_dir = prepare_image_directory(data_dir, DEFAULT_EXTENSIONS)
requested_split = Path(SPLIT_PATH).expanduser() if SPLIT_PATH else project_dir / "data/splits/clean_32x32"
split_dir = (requested_split if requested_split.is_absolute() else project_dir / requested_split).resolve()
expected_classes, label_level = EXPECTED_CLASSES, LABEL_LEVEL
required = ["train.csv", "val.csv", "label_to_index.json", "split_meta.json"]
if not all((split_dir / f).is_file() for f in required):
    raise FileNotFoundError("ต้องรับ split กลางครบ 4 ไฟล์ก่อน: " + str(split_dir))
if (data_dir / ".download_incomplete").exists():
    raise RuntimeError("dataset ยังดาวน์โหลดไม่ครบ")
if not any(image_paths(data_dir, DEFAULT_EXTENSIONS)) and not any(data_dir.glob("*.zip")):
    if not DOWNLOAD_IF_MISSING:
        raise FileNotFoundError("ไม่พบข้อมูลจริง: ตั้ง DATA_PATH หรือเปิด DOWNLOAD_IF_MISSING")
    if data_dir.exists() and any(data_dir.iterdir()):
        raise RuntimeError("โฟลเดอร์มีไฟล์แต่ไม่พบภาพหรือ ZIP: ตรวจ DATA_PATH ก่อนดาวน์โหลด")
    try:
        subprocess.run([sys.executable, "-m", "src.download_data", "--output-dir", str(data_dir)],
                       cwd=project_dir, check=True)
    except BaseException:
        data_dir.mkdir(parents=True, exist_ok=True)
        (data_dir / ".download_incomplete").touch()
        raise

audit_dir = results_dir / MODE / "audit"
audit_report = audit_dataset(data_dir, audit_dir, label_level, DEFAULT_EXTENSIONS)
display(pd.Series({k: audit_report[k] for k in ["valid_images", "class_count", "corrupt_images", "label_errors", "exact_duplicate_groups"]}))
if audit_report["corrupt_images"] or audit_report["label_errors"]:
    raise ValueError("แก้ corrupt images / labels ก่อนทดลอง")
# ใช้ settings ของ split กลาง ไม่ผูก split seed กับ training seed
previous_meta = json.loads((split_dir / "split_meta.json").read_text()) if (split_dir / "split_meta.json").exists() else {}
split_meta = prepare_split(data_dir, audit_dir, split_dir,
    seed=previous_meta.get("seed", 42), val_fraction=previous_meta.get("val_fraction", 0.2),
    expected_classes=expected_classes,
    conflicting_label_policy=previous_meta.get("conflicting_label_policy", "exclude"))
train_rows, val_rows, mapping, _ = load_split(data_dir, split_dir)
if split_meta.get("train_only_classes") or {r["label"] for r in val_rows} != set(mapping):
    print("คำเตือน: คลาสที่ไม่มี validation จะไม่ถูกวัดจากภาพ validation: " + str(split_meta.get("train_only_classes", [])))
counts = pd.crosstab(pd.Series([r["label"] for r in train_rows + val_rows], name="label"),
                     ["train"] * len(train_rows) + ["validation"] * len(val_rows))
display(counts)

print("✅ REAL DATA พร้อมใช้งาน")
print("Dataset root:", data_dir)
print("โฟลเดอร์ภาพจริง:", data_dir)
print("จำนวนภาพ / คลาส:", audit_report["valid_images"], "/", audit_report["class_count"])

fig, axes = plt.subplots(2, 6, figsize=(12, 5))
sample_rows = pd.DataFrame(train_rows).sort_values(["label", "path"]).groupby("label").head(1).head(12)
for axis, row in zip(axes.flat, sample_rows.itertuples()):
    with Image.open(data_dir / row.path) as image:
        axis.imshow(image.convert("RGB"), cmap="gray")
    axis.set_title(f"label: {row.label}")
for axis in axes.flat:
    axis.axis("off")
plt.suptitle("REAL DATA samples before preprocessing")
plt.tight_layout()
plt.savefig(audit_dir / "dataset_samples.png", dpi=140)
plt.show()

## 3. บันทึกสูตรทดลอง

โมเดล transfer ใช้ pretrained weights, freeze 2 epochs, seed 42, input 224, batch 32,
head LR 0.001, fine-tune LR 0.0001, dropout 0.3 และไม่เปิด augmentation/class weights
Custom CNN ฝึกจากศูนย์ สูตรและ weight tag จริงจะอยู่ใน `protocol.json` และ config ของแต่ละ run
โฟลเดอร์ผลแยกตามทีม/สมาชิก/โหมด/split/สูตร เพื่อไม่เขียนทับงานเพื่อน

In [ ]:
def resolve_candidate(row):
    result = dict(row)
    name = row["model"]
    if name == "custom_cnn":
        return {**result, "architecture": name, "status": "ready", "pretrained_recipe": {}}
    if not timm.is_model(name):
        return {**result, "status": "unavailable", "error": "model absent from installed timm"}
    cfg = timm.models.get_pretrained_cfg(name)
    if cfg is None or not cfg.has_weights:
        return {**result, "status": "unavailable", "error": "no registered pretrained weights"}
    name = name if "." in name or not cfg.tag else f"{name}.{cfg.tag}"
    recipe = cfg.to_dict()
    return {**result, "architecture": name, "status": "ready", "pretrained_recipe": recipe,
            "native_size": list(cfg.input_size), "native_mean": list(cfg.mean), "native_std": list(cfg.std)}

catalog = [resolve_candidate(row) for row in selected.to_dict("records")]
base_config = TrainConfig(seed=SEED, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, epochs=epochs,
                          freeze_epochs=2, pretrained=True, device=DEVICE, pad_value=PAD_VALUE)
versions = {name: importlib.metadata.version(name) for name in
            ["torch", "torchvision", "timm", "numpy", "scikit-learn", "optuna"]}
contract = {"owner": OWNER, "team_size": TEAM_SIZE, "member_id": MEMBER_ID, "round": ROUND,
            "job_id": assigned_job_id, "assignment_mode": assignment_mode, "mode": MODE,
            "stage": "family_screening", "config": asdict(base_config), "catalog": catalog,
            "split_hash": split_meta["split_hash"], "versions": versions,
            "trainer_hash": fingerprint([(project_dir / "src" / f).read_text()
                                         for f in ["train.py", "split.py", "search.py"]])}
experiment_dir = results_dir / MODE / split_meta["split_hash"][:12] / fingerprint(contract)[:12]
experiment_dir.mkdir(parents=True, exist_ok=True)
save_json(experiment_dir / "protocol.json", contract)
pd.DataFrame(catalog).to_csv(experiment_dir / "candidate_catalog.csv", index=False)

active_config = asdict(base_config)
active_config["architecture"] = catalog[0].get("architecture", catalog[0]["model"])
active_config["pretrained"] = base_config.pretrained and catalog[0]["model"] != "custom_cnn"
print("โมเดลที่เลือก:", catalog[0]["family"], "->", active_config["architecture"])
display(pd.Series(active_config, name="Training config ที่ใช้กับงานนี้"))
print("Output:", experiment_dir)

## 4. เทรนโมเดลของงานนี้และบันทึกผล

หนึ่งครั้งของ Run All เทรนเพียงโมเดลเดียวที่เลือกใน Cell 5 โดยตรวจ forward/backward ก่อนฝึก

ผลสำเร็จอยู่ใน `screening.csv` และสถานะอยู่ใน `screening_status.csv` รันงานเดิมซ้ำได้โดย run ที่เสร็จแล้วจะ reuse

หากโมเดลล้มเหลว notebook ยังสร้าง `handoff.json` โดยระบุ `complete=false` ให้ส่งสถานะนี้ผู้รวมผล แล้วแก้สาเหตุหรือให้คนอื่นรับ `JOB_ID` เดิมใหม่ ห้ามแทนคะแนนที่หายด้วยศูนย์

หาก OOM อย่าลด batch เฉพาะงานแล้วนำไปเทียบกับ config เดิม ให้ตกลงสูตรใหม่ร่วมกันหรือย้ายงานไปเครื่องที่รองรับ

In [ ]:
def preflight(config, classes):
    device = choose_device(config.device)
    model = None
    try:
        model = build_model(config, classes, load_pretrained=False).to(device)
        count = sum(p.numel() for p in model.parameters())
        # head-only pass ตรงกับ freeze stage; full pass ตรวจ training output ด้วย
        for frozen in (True, False):
            for p in model.parameters():
                p.requires_grad_(not frozen)
            classifier = model.get_classifier()
            if not isinstance(classifier, torch.nn.Module) or not list(classifier.parameters()):
                raise TypeError("trainer requires a trainable get_classifier() module")
            for p in classifier.parameters():
                p.requires_grad_(True)
            model.eval() if frozen else model.train()
            images = torch.randn(2, 3, config.image_size, config.image_size, device=device)
            logits = model(images)
            if not isinstance(logits, torch.Tensor) or logits.shape != (2, classes):
                raise ValueError("model must return a single [batch, classes] tensor")
            loss = torch.nn.functional.cross_entropy(logits, torch.tensor([0, classes - 1], device=device))
            if not torch.isfinite(loss):
                raise FloatingPointError("non-finite preflight loss")
            loss.backward()
            grads = [p.grad for p in model.parameters() if p.grad is not None]
            if not grads or not all(torch.isfinite(g).all().item() for g in grads):
                raise FloatingPointError("missing/non-finite gradients")
            model.zero_grad(set_to_none=True)
        return count
    finally:
        del model
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
        elif device.type == "mps":
            torch.mps.empty_cache()

receipts, statuses = [], []
preflight_cache_path = experiment_dir / "preflight.json"
preflight_cache = json.loads(preflight_cache_path.read_text()) if preflight_cache_path.exists() else {}
for candidate in catalog:
    status = {k: candidate[k] for k in ["family", "model", "status"]}
    if candidate["status"] != "ready":
        status["error"] = candidate.get("error", "unavailable")
    else:
        config = replace(base_config, architecture=candidate["architecture"],
                         pretrained=base_config.pretrained and candidate["model"] != "custom_cnn")
        try:
            key = fingerprint([asdict(config), len(mapping), str(choose_device(DEVICE))])
            if key not in preflight_cache:
                preflight_cache[key] = preflight(config, len(mapping))
                save_json(preflight_cache_path, preflight_cache)
            status["parameter_count"] = preflight_cache[key]
            receipt = fit(config, data_dir, split_dir, experiment_dir / "runs", phase="family_screening")
            receipts.append({**receipt, "family": candidate["family"], "group": candidate["group"],
                             "hypothesis": candidate["hypothesis"]})
            status.update(status="complete", run_id=receipt["run_id"])
        except Exception as error:
            status.update(status="failed", error=f"{type(error).__name__}: {error}")
            print("FAILED", candidate["family"], status["error"])
    statuses.append(status)
    pd.DataFrame(statuses).to_csv(experiment_dir / "screening_status.csv", index=False)
    if receipts:
        rank_runs(receipts).to_csv(experiment_dir / "screening.csv", index=False)

if receipts:
    screening = rank_runs(receipts)
    screening["train_val_gap"] = screening.train_accuracy - screening.val_accuracy
else:
    screening = pd.DataFrame(columns=[
        "family", "backbone", "hypothesis", "val_macro_f1", "val_accuracy",
        "minimum_per_class_recall", "parameter_count", "training_seconds",
        "train_accuracy", "train_val_gap", "split_hash", "seed", "checkpoint_path",
    ])
screening.to_csv(experiment_dir / "screening.csv", index=False)
display(pd.DataFrame(statuses))
if screening.empty:
    print("งานนี้ยังไม่มีโมเดลสำเร็จ ดู screening_status.csv และส่ง handoff เพื่อรับงานใหม่ภายหลัง")
else:
    display(screening[["family", "backbone", "val_macro_f1", "val_accuracy", "minimum_per_class_recall",
                       "parameter_count", "training_seconds", "train_val_gap"]])

## 5. ดูภาพจริงกับคำทำนาย

รายงานใช้ validation ชุดเดียวกับการเลือก checkpoint จึงไม่ใช่ unseen test
เมื่อมีผลเดิม ตั้ง `REVIEW_TABLE` แล้วรันส่วนตั้งค่า/imports/ข้อมูลและ cell นี้เพื่อดูผลโดยไม่เทรนใหม่

In [ ]:
from IPython.display import HTML
from src.visualize import validation_gallery

REVIEW_TABLE = ""  # ใส่ path CSV เพื่ออ่านผลเดิม โดยไม่รัน training cells
DISPLAY_LABELS = {}  # mapping รหัสคลาส -> อักษรไทย ที่ตรวจแล้ว; ว่าง = แสดง label เดิม
VISUAL_SAMPLE_COUNT = 8
if REVIEW_TABLE:
    review_path = Path(REVIEW_TABLE).expanduser().resolve()
    review_receipts = pd.read_csv(review_path).to_dict("records")
    review_dir = review_path.parent / (review_path.stem + "_visuals")
    review_context = "ผลที่บันทึกไว้: " + str(review_path)
else:
    review_receipts = screening.to_dict("records")
    review_dir = experiment_dir / "screening_visuals"
    review_context = MODE

if not review_receipts:
    print("ข้ามรายงานภาพ: งานนี้ยังไม่มี checkpoint สำเร็จ")
else:
    if any(r["split_hash"] != split_meta["split_hash"] for r in review_receipts):
        raise ValueError("รายงานกับ split ปัจจุบันไม่ตรงกัน ให้โหลด split ของรอบนั้น")
    visual_html = validation_gallery(review_receipts, data_dir, val_rows, review_dir,
        seed=SEED, sample_count=VISUAL_SAMPLE_COUNT, labels=DISPLAY_LABELS, context=review_context)
    display(HTML(visual_html))
    print("เปิดรายงานได้โดยไม่เทรนใหม่:", review_dir / "index.html")
    display(pd.DataFrame(review_receipts)[["backbone", "seed", "checkpoint_path"]].reset_index(names="model_index"))

## 6. ส่งงานและรับงานถัดไป

ส่ง **โฟลเดอร์ `experiment_dir` ทั้งโฟลเดอร์** ให้ผู้รวมผล รวม `runs/`, `handoff.json`, `protocol.json`, `screening.csv`, `screening_status.csv` และ `experiment_notes.csv`

ผู้รวมผลใช้ `job_id` ตรวจงาน 1–41: งานสำเร็จต้องมี `complete=true` และ checkpoint จริง งาน failed ต้องถูกเปิดให้แก้หรือรับใหม่ก่อนสรุปผล อย่านับเพียงจำนวนไฟล์ เพราะ `JOB_ID` อาจถูกเลือกซ้ำได้ถ้าไม่ได้จองในตารางแชร์

เมื่อส่งงานแล้ว ให้เพิ่ม `ROUND` เพื่อรับงานถัดไป หรือเปลี่ยน `JOB_ID` เป็นเลขงานว่างที่ทีมตกลงกัน จากนั้น Restart Kernel และ Run All ใหม่ ผลจะลงคนละโฟลเดอร์ `job-XX`

ก่อนรวมคะแนนต้องตรวจว่า split hash, สูตรฝึก, mode, versions และ trainer hash ตรงกัน แล้วรวมครบ 41 งานก่อนเลือก shortlist ของทั้งทีม อย่าใช้เวลาจาก GPU ต่างเครื่องจัดอันดับความเร็ว

ผลเป็น validation สำหรับเลือกโมเดล ยังต้องประเมิน unseen test ก่อนสรุปผลสุดท้าย path ใน receipt อาจชี้เครื่องเดิมเมื่อย้ายไฟล์ จึงต้องปรับ artifact paths ก่อนโหลด checkpoint บนอีกเครื่อง

In [ ]:
notes_path = experiment_dir / "experiment_notes.csv"
if not notes_path.exists():
    notes = screening[["family", "backbone", "hypothesis", "val_macro_f1", "val_accuracy",
                       "minimum_per_class_recall", "parameter_count", "training_seconds"]].copy()
    notes["owner"] = OWNER
    notes["job_id"] = assigned_job_id
    notes["round"] = ROUND
    for column in ["confused_pairs", "conclusion", "next_experiment"]:
        notes[column] = ""
    notes.to_csv(notes_path, index=False)
failed = [row for row in statuses if row["status"] != "complete"]
save_json(experiment_dir / "handoff.json", {
    "owner": OWNER, "team_size": TEAM_SIZE, "member_id": MEMBER_ID, "round": ROUND,
    "job_id": assigned_job_id, "assignment_mode": assignment_mode,
    "mode": MODE, "stage": "family_screening", "split_hash": split_meta["split_hash"],
    "assigned_families": [row["family"] for row in catalog],
    "completed_families": screening.family.tolist(), "failed": failed,
    "complete": not failed, "protocol": "protocol.json", "screening": "screening.csv",
})
print("งานหมายเลข:", assigned_job_id, "ครบ:", not failed)
print("เติมข้อสรุป:", notes_path)
print("ส่งทั้งโฟลเดอร์:", experiment_dir)
if failed:
    display(pd.DataFrame(failed))